# Basic Chatbot
Create a basic, interactive CLI chatbot using a causal language model from Hugging Face. We use the `transformers` library for model and tokenizer loading, and `bitsandbytes` to load the model efficiently in 4-bit precision.

In [1]:
!pip install -q -U torch transformers accelerate bitsandbytes huggingface_hub ipywidgets

In [2]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList
)
from huggingface_hub import login

## Authentication
Need to authenticate with the Hugging Face Hub using a private access token (`hf_token`) to download and access restricted models, such as Llama-3.

In [3]:

# Variable: String containing your Hugging Face API token for authentication.
hf_token = '--- Huggingface API key ---'

try:
    if not hf_token:
        raise ValueError("HF_TOKEN not found. Please set it and rerun.")
    print("Logging into Hugging Face...")
    login(token=hf_token)
except ValueError as ve:
    print(ve)
except Exception as e:
    print(f"Login failed: {e}")


Logging into Hugging Face...
Login failed: Invalid user token.


## Model Initialization
Here we define our target model. We use `BitsAndBytesConfig` to quantize the model to 4 bits (`load_in_4bit=True`), significantly reducing the VRAM required to run it without severely impacting quality. The tokenizer translates text strings into token IDs, and we instantiate the causal language model for generation.

In [4]:
# Variable: String specifying the Hugging Face Model architecture/ID we want to load.
model_id = "meta-llama/Llama-3.1-8B-Instruct"

print(f"\nInitializing {model_id}...")

# Configure 4-bit quantization
# Variable: BitsAndBytesConfig configuration specifying 4-bit precision to save VRAM.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading tokenizer...")
# Variable: The tokenizer object responsible for transforming raw strings into neural net tokens.
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

# Pad correctly to avoid pad_token_id warnings
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    
print("Loading model (this might take a few minutes)...")
# Variable: The transformer causal language model object initialized in 4-bit mode.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16,
)


Initializing meta-llama/Llama-3.1-8B-Instruct...
Loading tokenizer...
Loading model (this might take a few minutes)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## Custom Stopping Criteria
In a conversational context, LLMs might try to simulate the user's side of the conversation by generating things like `\nUser:`. To prevent this, we create a custom `StopOnSubsequence` class that continuously monitors the generated tokens and halts the model automatically as soon as it attempts to output reserved tags.

In [5]:
print("Setting up stopping criteria...")
# Stop generation when it tries to start a new speaker turn
# Variable: A list of strings that trigger the chatbot generation to stop simulating turns.
STOP_STRINGS = ["\nUser:", "\nSYSTEM:", "\nAssistant:"]

# Function: Helper to explicitly encode a single string into standard token IDs.
# Parameter (s): The string to tokenize.
def _ids(s: str):
    return tokenizer(s, add_special_tokens=False)["input_ids"]
    
# Variable: A list of the token IDs mapped from the STOP_STRINGS.
STOP_IDS_LIST = [ids for ids in (_ids(s) for s in STOP_STRINGS) if ids]

# Class: A custom StopCriteria that halts text generation immediately when a specified subsequence of token IDs is generated.
class StopOnSubsequence(StoppingCriteria):
    # Method: Constructor for the Custom StopCriteria sequence check.
    # Parameter (stop_ids_list): List of integers. Evaluated when checking.
    def __init__(self, stop_ids_list):
        self.stop_ids_list = [torch.tensor(x) for x in stop_ids_list]
        
    # Method: The main call method that gets executed on every newly generated token block to decide if stopping should occur.
    def __call__(self, input_ids, scores, **kwargs):
        for stop_ids in self.stop_ids_list:
            n = len(stop_ids)
            if input_ids.shape[1] >= n:
                if torch.all(input_ids[0, -n:] == stop_ids.to(input_ids.device)):
                    return True
        return False
        
# Variable: A StoppingCriteriaList structure grouping all defined stopping components to pass into our model.
stopping = StoppingCriteriaList([StopOnSubsequence(STOP_IDS_LIST)])


Setting up stopping criteria...


## System Prompt and Generation Logic
We set a `SYSTEM_PROMPT` to enforce strict instructions and define the behavior of the chatbot (in this case, it acts as a Jira Ticket Formatter).

The `base_chat_once` function handles:
1. Truncating the context slightly to prevent Out-Of-Memory errors over long conversations.
2. Concatenating user input with the running transcript.
3. Generating the assistant's reply using tokenizers and model generation.
4. Cleaning up any unintentional role-leakage not caught by stopping criteria early enough.

In [6]:

# Clearer SYSTEM prompt and role anchoring
# Variable: A static multi-line string defining the rules the system and assistant must abide by.
SYSTEM_PROMPT = ''' 
    SYSTEM: You are a strict Jira Ticket Formatter.
    SYSTEM RULES:
    - Your goal is to help the user write detailed, well-structured Jira tickets (Bugs, Tasks, Stories, or Epics).
    - Must include (Title:, Type:, Priority:, Description:, Labels:, Attachments:, Reporter:, Acceptance Criteria:).
    - Be concise, direct, highly structured and professional.
    - CRITICAL RULE: DO NOT ask follow-up questions.
    - CRITICAL RULE: DO NOT output any multiple-choice options.
'''

# Function: Base generator that manages the chat transcript, creates prompt format and produces inference logic.
# Parameter (user_text): The string containing the query input by the user on standard input.
# Parameter (transcript): The complete history log of the current chat interaction state.
def base_chat_once(user_text, transcript: str):
    # Keeps context bounded (prevents slowing down over long chats)
    # Variable: Integer capping the conversation history length before truncation happens to avoid slowing down context input padding.
    MAX_CHARS = 6000
    if len(transcript) > MAX_CHARS:
        transcript = transcript[-MAX_CHARS:]
        
    # Append the user's input to the transcript
    transcript += f"\nUser: {user_text}\nAssistant:"
    
    # Tokenize
    # Variable: Key-value mapped tensor dictionary passed straight into the model inference cycle representing tokenized state arrays.
    inputs = tokenizer(transcript, return_tensors="pt").to(model.device)
    
    print("Bot: ", end="", flush=True)
    # Generate response
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=True,
            temperature=0.4,           # Less random decoding
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping # Prevents runaway dialogue
        )
        
    # Extract new tokens and decode
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    assistant_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # Clean up role leakage (if stopping criteria caught it a bit late)
    assistant_text = assistant_text.split("\nUser:")[0]
    assistant_text = assistant_text.split("\nSYSTEM:")[0]
    assistant_text = assistant_text.replace("Assistant:", "").strip()
    
    print(assistant_text)
    
    # Update transcript
    transcript += " " + assistant_text
    return assistant_text, transcript

print("\n" + "="*50)
print("Chatbot is ready! Type 'exit' or 'quit' to end the conversation.")
print("="*50 + "\n")

# Initialize the conversation transcript with the system prompt
# Variable: String holding the continuous chat history. Begins pre-populated with the system persona.
transcript = SYSTEM_PROMPT



Chatbot is ready! Type 'exit' or 'quit' to end the conversation.



## Interactive Chat Session
This runs a continuous `while True` loop to collect user input, pass it to the bot, and print the assistant's response. You can type `'exit'` or `'quit'` to gracefully end the conversation.

In [7]:

while True:
    try:
        user_input = input("You: ").strip()
        print(f"\nInput: {user_input}\n")
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot session ended. Goodbye!")
            break
            
        if not user_input:
            continue

        _, transcript = base_chat_once(user_input, transcript)
        print() # Blank line for spacing
        
    except KeyboardInterrupt:
        print("\nChatbot session ended by user. Goodbye!")
        break
    except Exception as e:
        print(f"\nError: {e}\n")  


Input: The checkout page is crashing when I try to buy the new Assassin's Creed DLC.

Bot: Type: Bug
Priority: High
Description: The checkout page crashes when attempting to purchase the new Assassin's Creed DLC. This prevents users from completing transactions.
Labels: #AssassinsCreed #DLC #CheckoutIssue
Attachments: [attach screenshot of crash]
Reporter: @username
Acceptance Criteria: 
1. The checkout page loads without crashing.
2. Users can successfully complete transactions for the new Assassin's Creed DLC.
3. Crash logs are reviewed to identify root cause.

Would you like me to add anything else?


Input: Assign "Don John" as reporter and change Priority to "Medium"

Bot: Type: Bug
Priority: Medium
Description: The checkout page crashes when attempting to purchase the new Assassin's Creed DLC. This prevents users from completing transactions.
Labels: #AssassinsCreed #DLC #CheckoutIssue
Attachments: [attach screenshot of crash]
Reporter: Don John
Acceptance Criteria: 
1. The chec